## EQUIPE

**Integrante 01:** Guilherme Araujo dos Santos — RA: 11202020294

**Integrante 02:** Ian Victor Toniolo Silva — RA: 11202020351


## GRANDE MODELO DE LINGUAGEM (Large Language Model - LLM)

**LLM:** Claude Haiku 5.5 (Anthropic) — identificador na API: `claude-haiku-5-5`

**Link para a documentação oficial:** https://docs.claude.com/en/docs/about-claude/models/overview

## API (Opcional)

**API:** MediaWiki Action API (Wikipédia em português)

**Site oficial:** https://www.mediawiki.org

**Link para a documentação oficial:** https://www.mediawiki.org/wiki/API:Main_page

## IMPLEMENTAÇÃO

### Visão geral
O usuário informa um **tema**. O notebook:
1. Busca o artigo mais relevante na **Wikipédia** via **MediaWiki Action API** (`list=search` + `prop=extracts`);
2. Usa **LangChain** + **Claude Haiku 5.5** para aplicar três técnicas de PLN sobre o texto do artigo:
   - **Sumarização de Textos**
   - **Reconhecimento de Entidades Nomeadas (NER)**, com saída estruturada (Pydantic)
   - **Sistema de Perguntas e Respostas** baseado no artigo (o modelo responde só com o que está no texto recuperado — uma forma simples de RAG)

### 1. Instalação das dependências

In [ ]:
# langchain-anthropic: integração do LangChain com os modelos Claude
# requests: chamadas HTTP para a MediaWiki API
!pip install -q langchain langchain-anthropic requests

### 2. Configuração da chave da API da Anthropic
A chave **não** fica no código. No Colab, clique no ícone de chave (🔑 *Secrets*) na barra lateral, crie um segredo chamado `ANTHROPIC_API_KEY` com a sua chave do Claude Console e habilite o acesso do notebook.

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("Chave carregada:", bool(os.environ["ANTHROPIC_API_KEY"]))

### 3. Coleta do córpus: MediaWiki Action API
Duas chamadas ao endpoint `https://pt.wikipedia.org/w/api.php`:
- `action=query&list=search` → encontra o título do artigo mais relevante para o tema;
- `action=query&prop=extracts&explaintext=1` → baixa o conteúdo do artigo em texto puro.

A Wikimedia exige um cabeçalho `User-Agent` identificando o cliente.

In [ ]:
import requests

WIKI_API = "https://pt.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "PLN-UFABC-ProjetoPratico/1.0 (projeto academico)"}


def buscar_titulo(tema: str) -> str:
    """Retorna o título do artigo da Wikipédia mais relevante para o tema."""
    params = {
        "action": "query",
        "list": "search",
        "srsearch": tema,
        "srlimit": 1,
        "format": "json",
    }
    resp = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    resultados = resp.json()["query"]["search"]
    if not resultados:
        raise ValueError(f"Nenhum artigo encontrado para: {tema}")
    return resultados[0]["title"]


def obter_texto(titulo: str, max_chars: int = 15000) -> str:
    """Baixa o texto puro do artigo (limitado a max_chars para controlar custo/tokens)."""
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": titulo,
        "redirects": 1,
        "format": "json",
    }
    resp = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    paginas = resp.json()["query"]["pages"]
    texto = next(iter(paginas.values())).get("extract", "")
    return texto[:max_chars]


# Teste rápido
titulo = buscar_titulo("Universidade Federal do ABC")
artigo = obter_texto(titulo)
print("Artigo:", titulo)
print("Tamanho:", len(artigo), "caracteres\n")
print(artigo[:800])

### 4. LLM via LangChain
`ChatAnthropic` é o wrapper do LangChain para os modelos Claude. `temperature=0` deixa as respostas mais determinísticas, o que é bom para tarefas de análise.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-haiku-5-5", temperature=0, max_tokens=1024)

# Teste de conexão
print(llm.invoke("Responda em uma frase: o que é PLN?").content)

### 5. Técnica 1: Sumarização de Textos
Uma *chain* LCEL (`prompt | llm | parser`): o prompt recebe o artigo e pede um resumo em português, em tópicos.

In [ ]:
prompt_resumo = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente que resume artigos enciclopédicos em português do Brasil, de forma fiel ao texto."),
    ("human", "Resuma o artigo abaixo em até 5 tópicos curtos.\n\nTítulo: {titulo}\n\nArtigo:\n{artigo}"),
])

chain_resumo = prompt_resumo | llm | StrOutputParser()

resumo = chain_resumo.invoke({"titulo": titulo, "artigo": artigo})
print(resumo)

### 6. Técnica 2: Reconhecimento de Entidades Nomeadas (NER)
Usamos `with_structured_output` com um esquema **Pydantic**: o LangChain força o modelo a devolver uma lista de entidades já tipadas (PESSOA, ORGANIZAÇÃO, LOCAL, DATA, OUTRO), pronta para virar tabela.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field
import pandas as pd


class Entidade(BaseModel):
    texto: str = Field(description="A entidade exatamente como aparece no texto")
    tipo: Literal["PESSOA", "ORGANIZACAO", "LOCAL", "DATA", "OUTRO"]


class ListaEntidades(BaseModel):
    entidades: List[Entidade] = Field(description="Entidades nomeadas distintas encontradas no texto")


prompt_ner = ChatPromptTemplate.from_messages([
    ("system", "Você é um sistema de Reconhecimento de Entidades Nomeadas. Extraia as entidades mais relevantes do texto, sem repetições."),
    ("human", "Extraia até 25 entidades nomeadas do texto abaixo.\n\n{artigo}"),
])

chain_ner = prompt_ner | llm.with_structured_output(ListaEntidades)

resultado_ner = chain_ner.invoke({"artigo": artigo})
df_entidades = pd.DataFrame([e.model_dump() for e in resultado_ner.entidades])
df_entidades.sort_values("tipo").reset_index(drop=True)

In [ ]:
# Contagem de entidades por tipo
df_entidades["tipo"].value_counts()

### 7. Técnica 3: Sistema de Perguntas e Respostas
O artigo recuperado da API é injetado no prompt como **contexto**, e o modelo é instruído a responder **apenas** com base nele (Geração Aumentada por Recuperação simples). Se a resposta não estiver no artigo, ele deve dizer isso em vez de inventar.

In [ ]:
prompt_qa = ChatPromptTemplate.from_messages([
    ("system",
     "Responda à pergunta usando SOMENTE o contexto fornecido (um artigo da Wikipédia). "
     "Se a resposta não estiver no contexto, responda: 'O artigo não traz essa informação.' "
     "Responda em português, de forma direta."),
    ("human", "Contexto:\n{artigo}\n\nPergunta: {pergunta}"),
])

chain_qa = prompt_qa | llm | StrOutputParser()

perguntas = [
    "Em que ano a universidade foi criada?",
    "Em quais cidades ficam os campi?",
    "Quem ganhou a Copa do Mundo de 2002?",  # fora do contexto: deve recusar
]

for p in perguntas:
    print("P:", p)
    print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")

### 8. Pipeline completo: qualquer tema
Junta tudo em uma função: tema → Wikipédia → resumo + entidades + respostas.

In [ ]:
def analisar_tema(tema: str, perguntas: list[str]):
    titulo = buscar_titulo(tema)
    artigo = obter_texto(titulo)
    print(f"=== Artigo: {titulo} ({len(artigo)} caracteres) ===\n")

    print("--- RESUMO ---")
    print(chain_resumo.invoke({"titulo": titulo, "artigo": artigo}), "\n")

    print("--- ENTIDADES NOMEADAS ---")
    ents = chain_ner.invoke({"artigo": artigo}).entidades
    display(pd.DataFrame([e.model_dump() for e in ents]))

    print("\n--- PERGUNTAS E RESPOSTAS ---")
    for p in perguntas:
        print("P:", p)
        print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")


analisar_tema("Alan Turing", [
    "Onde Alan Turing nasceu?",
    "Qual foi a contribuição dele durante a Segunda Guerra Mundial?",
])

### 9. Conclusão
- **API:** a MediaWiki Action API forneceu o córpus (artigos da Wikipédia em português) sob demanda, a partir de qualquer tema.
- **LangChain:** usado para montar as *chains* (prompt → LLM → parser) e para obter saída estruturada com Pydantic.
- **LLM:** o Claude Haiku 5.5 aplicou três técnicas de PLN: sumarização, NER e perguntas e respostas fundamentadas no texto recuperado.